In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import data
from check_submission import check_all

ROOT = Path.cwd()
TRACKS = ["off_market", "list_aware"]
# Walk-forward validation: each of these months is predicted by models trained only
# on the sales before that month, the same way the real test works.
VALIDATION_MONTHS = pd.period_range("2025-12", "2026-05", freq="M")


In [2]:
def validate(
    modelspecs,
    make_model_func=None,
    mode="expanding",
    train_cutoff=None,
    horizon_months=4,
):
    from models.model import make_model

    if make_model_func is None:
        make_model_func = make_model

    listings = data.load("listings")
    results = []
    model_list = []

    if mode == "expanding":
        # Original validation: retrain for every evaluation month
        months = VALIDATION_MONTHS

    elif mode == "fixed_origin":
        # Train once and freeze the model for the entire horizon
        cutoff = pd.Timestamp(train_cutoff)

        if cutoff != cutoff.to_period("M").start_time:
            raise ValueError("train_cutoff must be the first day of a month.")

        if horizon_months < 1:
            raise ValueError("horizon_months must be positive.")

        months = pd.period_range(
            start=cutoff.to_period("M"),
            periods=horizon_months,
            freq="M",
        )

        train = listings.loc[
            listings["date_sold"] < cutoff
        ]

        if train.empty:
            raise ValueError("No training sales before train_cutoff.")

        # Fit each track exactly ONCE
        fixed_models = [
            make_model_func(track, modelspecs=modelspecs).fit(train)
            for track in TRACKS
        ]

        model_list.append(fixed_models)

        print(
            f"Fixed origin: trained on {len(train):,} sales "
            f"before {cutoff.date()}"
        )

    else:
        raise ValueError("mode must be 'expanding' or 'fixed_origin'.")

    for month in months:
        homes = listings.loc[
            listings["date_sold"].dt.to_period("M") == month
        ]

        if mode == "expanding":
            train = listings.loc[
                listings["date_sold"] < month.start_time
            ]

            models = [
                make_model_func(track, modelspecs=modelspecs).fit(train)
                for track in TRACKS
            ]

            model_list.append(models)
        else:
            # Reuse the SAME fitted models for every month
            models = fixed_models

        for track, model in zip(TRACKS, models):
            predicted = model.predict(
                data.hide_answers(homes, track)
            )

            results.append(pd.DataFrame({
                "ml_num": homes["ml_num"],
                "track": track,
                "month": str(month),
                "house_type_name": homes["house_type_name"],
                "sold_price": homes["sold_price"],
                "predicted_price": np.round(predicted),
            }))

        if mode == "expanding":
            print(
                f"{month}: trained on {len(train):,} earlier sales, "
                f"predicted {len(homes):,} homes"
            )
        else:
            print(
                f"{month}: reused frozen models, "
                f"predicted {len(homes):,} homes"
            )

    return pd.concat(results, ignore_index=True), model_list

def bootstrap_ci(values, statistic, repeats=1000):
    """95% interval: recompute the statistic on 1,000 resamples of the homes
    (drawn with replacement) and keep the middle 95% of the results."""
    rng = np.random.default_rng(0)
    results = [statistic(rng.choice(values, size=len(values))) for _ in range(repeats)]
    return [round(float(x), 2) for x in np.percentile(results, [2.5, 97.5])]

def metrics(predictions):
    """Scores for one track. Errors are in percent of the sold price."""
    signed = 100 * (predictions["predicted_price"] - predictions["sold_price"]) / predictions["sold_price"]
    absolute = signed.abs().to_numpy()
    return {
        "n": len(predictions),
        "mape": round(float(np.mean(absolute)), 2),          # mean absolute % error
        "mape_95ci": bootstrap_ci(absolute, np.mean),
        "mdape": round(float(np.median(absolute)), 2),       # median absolute % error
        "mdape_95ci": bootstrap_ci(absolute, np.median),
        "median_signed_error": round(float(np.median(signed)), 2),  # above 0: predicts too high
    }
def report(df):
    # Accept results from earlier notebook runs without refitting the models.
    predictions = pd.concat(df, ignore_index=True) if isinstance(df, list) else df
    scores = {track: metrics(predictions[predictions["track"] == track]) for track in TRACKS}

    (ROOT / "reports").mkdir(exist_ok=True)
    (ROOT / "reports" / "validation_report.json").write_text(json.dumps(scores, indent=2))
    lines = [
        "# Validation report", "",
        f"Walk-forward validation, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}: each month was "
        "predicted by models trained only on the sales before it.",
        "Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.", "",
        "| Track | Homes | MAPE | MdAPE | Median signed error |", "|---|---|---|---|---|"]
    for track, s in scores.items():
        lines.append(f"| {track} | {s['n']:,} | {s['mape']:.2f}% ({s['mape_95ci'][0]:.2f} to {s['mape_95ci'][1]:.2f}) "
                     f"| {s['mdape']:.2f}% ({s['mdape_95ci'][0]:.2f} to {s['mdape_95ci'][1]:.2f}) "
                     f"| {s['median_signed_error']:+.2f}% |")
    lines += [
        "", "MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.",
        "A median signed error above 0 means the model tends to predict too high.",
        "The intervals treat homes as independent. Homes sold in the same month share the same",
        "market swings, so the real uncertainty is somewhat wider.", "",
        "![Error by price range](../figures/error_by_price_range.png)", ""]
    (ROOT / "reports" / "validation_report.md").write_text("\n".join(lines))
    write_latex_table(scores)
    error_by_price_range(predictions)
    print("\n".join(lines))
    print("Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex "
          "and figures/error_by_price_range.png")


def write_latex_table(scores):
    """The same scores as a LaTeX table, for report/report.tex to include."""
    rows = []
    for track, s in scores.items():
        name = track.replace("_", "-")
        rows.append(f"{name} & {s['n']:,} & {s['mdape']:.2f} ({s['mdape_95ci'][0]:.2f}--{s['mdape_95ci'][1]:.2f}) "
                    f"& {s['mape']:.2f} ({s['mape_95ci'][0]:.2f}--{s['mape_95ci'][1]:.2f}) "
                    f"& {s['median_signed_error']:+.2f} \\\\")
    table = "\n".join([
        "% Written by `python run.py report`. Do not edit by hand: rerun the command instead.",
        "\\begin{tabular}{lrccc}", "\\hline",
        "Track & Homes & MdAPE \\% (95\\% CI) & MAPE \\% (95\\% CI) & Median signed error \\% \\\\", "\\hline",
        *rows, "\\hline", "\\end{tabular}"])
    (ROOT / "reports" / "results_table.tex").write_text(table + "\n")


def error_by_price_range(predictions):
    """Figure: how big the errors are, from the cheapest tenth of homes to the dearest.

    Want more figures? Copy this function and group the errors by something else:
    house_type_name, month, or community (add it to the predictions file in validate()).
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colours = {"off_market": "#2a78d6", "list_aware": "#eb6834"}
    for track in TRACKS:
        rows = predictions[predictions["track"] == track]
        error = 100 * (rows["predicted_price"] - rows["sold_price"]).abs() / rows["sold_price"]
        decile = pd.qcut(rows["sold_price"], 10, labels=False) + 1   # 1 = cheapest tenth
        by_decile = error.groupby(decile)
        middle = by_decile.median()
        ax.plot(middle.index, middle.values, marker="o", linewidth=2, color=colours[track], label=track)
        ax.fill_between(middle.index, by_decile.quantile(0.25), by_decile.quantile(0.75),
                        color=colours[track], alpha=0.15, linewidth=0)
    prices = rows["sold_price"].groupby(decile).median()
    ax.set_xticks(prices.index, [f"{d}\n${p / 1e6:.2f}M" for d, p in prices.items()])
    ax.set_xlabel("Sold-price decile (1 = cheapest tenth of homes), with its median price")
    ax.set_ylabel("Absolute % error\n(line: median, band: middle 50%)")
    ax.set_title(f"Validation error by price range, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e5e5")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    (ROOT / "figures").mkdir(exist_ok=True)
    fig.savefig(ROOT / "figures" / "error_by_price_range.png", dpi=150)
    plt.close(fig)

In [3]:
from models.boost_regressor import CatBoostRegressor, LightGBMRegressor, XGBoostRegressor
from models.arima import ArimaResidualModel
from models.model import make_model
from models.direct_listing import DirectListingModel
from models.model import OffMarketModel

CATBOOST_PARAMS = {
    "iterations": 900,
    "depth": 6,
    "learning_rate": 0.04,
    "l2_leaf_reg": 5,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}

LIGHTGBM_PARAMS = {
    "n_estimators": 900,
    "learning_rate": 0.04,
    "max_depth": 6,
    "num_leaves": 31,
    "min_child_samples": 40,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "max_bin": 255,
    "objective": "regression",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

XGBOOST_PARAMS = {
    "n_estimators": 900,
    "learning_rate": 0.04,
    "max_depth": 6,
    "min_child_weight": 5,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
    "grow_policy": "depthwise",
    "device": "cpu",
    "enable_categorical": True,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}
0
LGB_MODEL_SPECS = [
    (LightGBMRegressor, "Xpt_spatiotemporal__selected_binary", {'n_estimators': 1800, 'learning_rate': 0.052995139011091236, 'max_depth': 6, 'num_leaves': 40, 'min_child_samples': 30, 'reg_lambda': 16.19372143031597, 'reg_alpha': 0.0, 'subsample': 0.8349167453037107, 'subsample_freq': 1, 'colsample_bytree': 0.953443325579757, 'max_bin': 255, 'objective': 'regression', 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}),
]
XGB_MODEL_SPECS = [
    (XGBoostRegressor, "Xpt_spatiotemporal__selected_binary", {'n_estimators': 700, 'learning_rate': 0.09522271657253033, 'max_depth': 8, 'min_child_weight': 16.684619386293466, 'reg_lambda': 19.50331236061207, 'reg_alpha': 0.0, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'device': 'cpu', 'enable_categorical': True, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}),
]
CAT_MODEL_SPECS = [
    (CatBoostRegressor, 'Xpt_spatiotemporal', {'iterations': 1400, 'depth': 8, 'learning_rate': 0.07127609366218435, 'l2_leaf_reg': 4.14053058084107, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'task_type': 'CPU', 'thread_count': -1, 'random_strength': 0.580328206016201}),
]
MODEL_SPECS = XGB_MODEL_SPECS
def custom_make_model(track, modelspecs=None):
    specs = MODEL_SPECS if modelspecs is None else modelspecs

    if track == "list_aware":
        return DirectListingModel(
            target="ratio",
            modelspecs=specs,
        )

    return OffMarketModel(
        specs,
        center_target=True,
    )


# Default to already-saved XGBoost model snapshots instead of fitting twice.
RUN_BASELINE_TRAINING = False
if RUN_BASELINE_TRAINING:
    predictions, models = validate(
        modelspecs=MODEL_SPECS,
        make_model_func=custom_make_model,
    )

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

import data
from save_model import load_model, model_info


def recover_validation(
    model_name="XGBoost_text_binary",
    save_predictions=True,
):
    # 1. Restore the fitted models
    models = load_model(model_name)
    info = model_info(model_name)

    months = pd.period_range("2025-12", "2026-05", freq="M")
    tracks = ["off_market", "list_aware"]

    # 2. Check that the saved models match our validation setup
    assert len(models) == len(months), "Unexpected number of saved months"
    assert all(len(pair) == 2 for pair in models), "Expected two tracks per month"

    metadata = info.get("metadata", {})
    if metadata.get("validation_months"):
        assert metadata["validation_months"] == [str(m) for m in months]
    if metadata.get("tracks"):
        assert metadata["tracks"] == tracks

    # 3. Load original validation properties
    listings = data.load("listings")
    sale_months = listings["date_sold"].dt.to_period("M")

    results = []

    # 4. Reconstruct predictions without fitting
    for month, month_models in zip(months, models):
        homes = listings.loc[sale_months == month]

        for track, model in zip(tracks, month_models):
            hidden = data.hide_answers(homes, track)
            predicted = np.round(model.predict(hidden))

            results.append(pd.DataFrame({
                "ml_num": homes["ml_num"].to_numpy(),
                "track": track,
                "month": str(month),
                "house_type_name": homes["house_type_name"].to_numpy(),
                "sold_price": homes["sold_price"].to_numpy(),
                "predicted_price": predicted,
            }))

        print(f"{month}: recovered {len(homes):,} homes")

    predictions = pd.concat(results, ignore_index=True)

    # 5. Save recovered predictions
    if save_predictions:
        path = Path("outputs/validation_predictions.csv")
        path.parent.mkdir(parents=True, exist_ok=True)
        predictions.to_csv(path, index=False)
        print(f"\nSaved predictions to {path}")

    return predictions, models


from models.spatiotemporal_features import ComparableSalesFeatures

# Restore original behaviour for legacy saved models
ComparableSalesFeatures.age_filter_first = False
ComparableSalesFeatures.min_same_type_neighbors = 0

# Restore the same variables used by the main notebook
if not RUN_BASELINE_TRAINING:
    predictions, models = recover_validation()
else:
    print("Using freshly fitted baseline models")

display(predictions.head())

2025-12: recovered 1,238 homes
2026-01: recovered 1,058 homes
2026-02: recovered 1,442 homes
2026-03: recovered 1,816 homes
2026-04: recovered 2,224 homes
2026-05: recovered 2,291 homes

Saved predictions to outputs\validation_predictions.csv


,ml_num,track,month,house_type_name,sold_price,predicted_price
0,SYN-0154350,off_market,2025-12,Condo Apt,797100.0,1016130.0
1,SYN-0150378,off_market,2025-12,Detached,1821500.0,1872584.0
2,SYN-0148591,off_market,2025-12,Condo Apt,711300.0,766904.0
3,SYN-0155236,off_market,2025-12,Condo Apt,257500.0,325986.0
4,SYN-0040814,off_market,2025-12,Condo Apt,516500.0,544869.0


In [7]:
report(predictions)

# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 10.38% (10.18 to 10.58) | 7.82% (7.64 to 7.99) | +0.41% |
| list_aware | 10,069 | 5.20% (5.12 to 5.29) | 4.17% (4.09 to 4.26) | +0.51% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means the model tends to predict too high.
The intervals treat homes as independent. Homes sold in the same month share the same
market swings, so the real uncertainty is somewhat wider.

![Error by price range](../figures/error_by_price_range.png)

Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex and figures/error_by_price_range.png


In [10]:
# Share these results with notebook.ipynb and the existing report command.
output_path = ROOT / "outputs" / "validation_predictions.csv"
output_path.parent.mkdir(exist_ok=True)
predictions.to_csv(output_path, index=False)

## Full-strength spatiotemporal individual-model comparison (off-market only)

This experiment keeps the original 700-tree XGBoost reference from `predictions, models`. No baseline retraining.

**Full-history refits:** `spacetime_splines` (3-month winner), `geo_context`, original `combined` (contains splines), `comp_type_quota` and `comp_both` (historical reference controls). Each has the **identical tuned estimator settings**, median-centered log-price target, same chronological training cutoff, same training records and held-out properties.

**Previously evaluated corrections:** `tensor_gam` and `rbf_ridge` are independently recalibrated using the three preceding months of truly out-of-time XGBoost errors. They correct **original baseline XGBoost**, not spline XGBoost. No combination is evaluated in this full-strength run.

The output contains independent per-variant/month predictions with restart-safe caching, pooled and monthly MAPE/MdAPE, paired bootstrap differences and house-type diagnostics. The new results live at `outputs/spacetime_full_feature_comparison/` and do not replace your original validation outputs or saved models.

By default `Run All` restores the trusted `saved_models/XGBoost_text_binary` snapshot rather than training a second XGBoost baseline. You can set `RUN_BASELINE_TRAINING=True` in cell 2 to retrain from scratch. The save step is opt-in so existing model files are not overwritten.

**Separate combination experiment:** `spacetime_combination_screen.ipynb` uses only 10k samples / 220 trees for spline + geo, spline + quota, spline + market, etc. before promoting any combination to the expensive final validation.

**Caution:** Dec 2025–May 2026 has already been consulted; treat this as confirmation rather than an untouched final test.


In [8]:
# Full-history: each feature and residual model evaluated individually.
RUN_FULL_FEATURE_COMPARISON = False  # previous full run already completed

if RUN_FULL_FEATURE_COMPARISON:
    from spacetime_full_feature_comparison import compare_full_spacetime_models
    from IPython.display import display

    full_pooled, full_monthly, full_by_type, full_predictions = compare_full_spacetime_models(
        baseline_predictions=predictions,
        fitted_models=models,
        months=VALIDATION_MONTHS,
        feature_variants=(
            "baseline",
            "spacetime_splines",
            "geo_context",
            "combined",
            "comp_type_quota",
            "comp_both",
        ),
        residual_variants=("tensor_gam", "rbf_ridge"),
        mode="expanding",
        calibration_months=3,
        calibration_cap=None,
        residual_strength=0.5,
        bootstrap_repeats=500,
        output_dir="outputs/spacetime_full_feature_comparison",
        resume=True,
    )
    display(full_pooled)
    display(full_monthly)
    display(full_by_type)
    print("Results saved in outputs/spacetime_full_feature_comparison/")



2025-12: 1,238 holdouts; full history before 2025-12-01
  spacetime_splines: fitting 700-tree spec to 139,931 sales
  geo_context: fitting 700-tree spec to 139,931 sales
  combined: fitting 700-tree spec to 139,931 sales
  comp_type_quota: fitting 700-tree spec to 139,931 sales
  comp_both: fitting 700-tree spec to 139,931 sales
    2025-09: fitting tuned base model on 133,779 earlier sales; predicting 2,035 unseen homes
    2025-10: fitting tuned base model on 135,814 earlier sales; predicting 2,250 unseen homes
    2025-11: fitting tuned base model on 138,064 earlier sales; predicting 1,867 unseen homes
  finished 2025-12

2026-01: 1,058 holdouts; full history before 2026-01-01
  spacetime_splines: fitting 700-tree spec to 141,169 sales
  geo_context: fitting 700-tree spec to 141,169 sales
  combined: fitting 700-tree spec to 141,169 sales
  comp_type_quota: fitting 700-tree spec to 141,169 sales
  comp_both: fitting 700-tree spec to 141,169 sales
    2025-12: reusing existing fitte

,homes,mape,mdape,median_signed,arm,variant,delta_mdape_pp,delta_mape_pp,mean_paired_ape_delta,homes_improved_fraction,delta_mdape_ci_low,delta_mdape_ci_high,delta_mape_ci_low,delta_mape_ci_high,months_better_mdape
0,10069,10.019250,7.588816,0.407211,feature,spacetime_splines,-0.229425,-0.362288,-0.362288,0.527858,-0.407554,-0.080800,-0.445519,-0.284421,6
1,10069,10.234494,7.621123,0.402116,feature,geo_context,-0.197119,-0.147043,-0.147043,0.516834,-0.336013,-0.050441,-0.223616,-0.068028,6
2,10069,10.356925,7.801095,0.304044,residual,tensor_gam,-0.017146,-0.024613,-0.024613,0.498957,-0.085074,0.064594,-0.041311,-0.007828,3
3,10069,10.267962,7.815925,0.483198,feature,combined,-0.002316,-0.113576,-0.113576,0.507101,-0.180808,0.162183,-0.209906,-0.020337,5
4,10069,10.381538,7.818242,0.408726,feature,baseline,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
5,10069,10.360145,7.850914,0.276124,residual,rbf_ridge,0.032672,-0.021393,-0.021393,0.499255,-0.044927,0.110743,-0.040671,-0.000111,2
6,10069,10.562967,8.088569,0.454084,feature,comp_type_quota,0.270327,0.181429,0.181429,0.483067,0.101366,0.428660,0.062996,0.275878,0
7,10069,10.647414,8.155765,0.309378,feature,comp_both,0.337523,0.265876,0.265876,0.479591,0.141122,0.491685,0.156103,0.370385,0


,homes,mape,mdape,median_signed,month,arm,variant,delta_mdape_pp,delta_mape_pp
0,1238,10.572629,7.953344,1.559994,2025-12,residual,rbf_ridge,-0.325383,-0.348547
1,1238,10.593873,7.954467,1.725276,2025-12,residual,tensor_gam,-0.324260,-0.327303
2,1238,10.617222,8.003067,2.997089,2025-12,feature,geo_context,-0.275660,-0.303953
3,1238,10.557862,8.184548,2.969956,2025-12,feature,spacetime_splines,-0.094179,-0.363314
4,1238,10.758780,8.249434,3.508063,2025-12,feature,combined,-0.029293,-0.162395
5,1238,10.921176,8.278727,2.916296,2025-12,feature,baseline,0.000000,0.000000
6,1238,11.073121,8.507402,3.476305,2025-12,feature,comp_type_quota,0.228675,0.151946
7,1238,11.220694,8.570544,3.411712,2025-12,feature,comp_both,0.291817,0.299518
8,1058,9.947284,7.394714,0.999829,2026-01,feature,spacetime_splines,-0.329683,-0.402283
9,1058,10.140594,7.536393,0.146382,2026-01,feature,combined,-0.188004,-0.208973


,homes,mape,mdape,median_signed,house_type_name,arm,variant,delta_mdape_pp,delta_mape_pp
0,48,18.583158,12.634448,-3.030243,Co-Op Apt,residual,rbf_ridge,-0.554036,0.136932
1,48,18.161121,12.787432,-2.975226,Co-Op Apt,feature,geo_context,-0.401052,-0.285104
2,48,18.579053,12.866223,-2.768359,Co-Op Apt,residual,tensor_gam,-0.322261,0.132828
3,48,19.935290,12.889698,-1.895429,Co-Op Apt,feature,comp_type_quota,-0.298785,1.489065
4,48,18.446225,13.188484,-1.188152,Co-Op Apt,feature,baseline,0.000000,0.000000
5,48,20.331036,13.301718,-2.328882,Co-Op Apt,feature,comp_both,0.113235,1.884811
6,48,18.689882,13.347487,0.973267,Co-Op Apt,feature,spacetime_splines,0.159003,0.243656
7,48,20.238399,14.393685,1.130542,Co-Op Apt,feature,combined,1.205201,1.792173
8,4843,8.763092,6.485410,1.303846,Condo Apt,feature,spacetime_splines,-0.307739,-0.430327
9,4843,8.968478,6.661043,1.287038,Condo Apt,feature,geo_context,-0.132105,-0.224940


Results saved in outputs/spacetime_full_feature_comparison/


In [ ]:
# Optional: keep the previously saved trusted baseline intact by default.
SAVE_BASELINE_MODELS = False
if SAVE_BASELINE_MODELS:
    from save_model import save_model
    save_model(
        models, "XGBoost_text_binary",
        metadata={
            "features": "Xpt_spatiotemporal__selected_binary",
            "tracks": list(TRACKS),
            "validation_months": [str(m) for m in VALIDATION_MONTHS],
            "mode": "expanding",
        },
    )


                            Homes   MAPE  MdAPE
track      horizon month                       
list_aware 1       2026-02   1442   5.44   4.31
           2       2026-03   1816   5.39   4.20
           3       2026-04   2224   5.35   4.31
           4       2026-05   2291   5.57   4.50
off_market 1       2026-02   1442  11.46   8.90
           2       2026-03   1816  11.53   8.84
           3       2026-04   2224  11.81   9.29
           4       2026-05   2291  12.27   9.69


Xpt -> BoostingFeatures
Xpt_spatiotemporal -> RatioComparableFeatures
Stages: 1
Feature provider: RatioComparableFeatures
Trained feature names:
['square_feet', 'bedrooms', 'washrooms', 'parking', 'garage_count', 'lot_front', 'lot_depth', 'lot_area', 'longitude', 'latitude', 'months_since_2021', 'season_sin', 'season_cos', 'log_list_price', 'house_type_name', 'house_style', 'municipality', 'community', 'garage', 'tfidf_0', 'tfidf_1', 'tfidf_2', 'tfidf_3', 'tfidf_4', 'tfidf_5', 'tfidf_6', 'tfidf_7', 'tfidf_8', 'tfidf_9', 'tfidf_10', 'tfidf_11', 'tfidf_12', 'tfidf_13', 'tfidf_14', 'tfidf_15', 'tfidf_16', 'tfidf_17', 'tfidf_18', 'tfidf_19', 'tfidf_20', 'tfidf_21', 'tfidf_22', 'tfidf_23', 'tfidf_24', 'tfidf_25', 'tfidf_26', 'tfidf_27', 'tfidf_28', 'tfidf_29', 'tfidf_30', 'tfidf_31', 'tfidf_32', 'tfidf_33', 'tfidf_34', 'tfidf_35', 'tfidf_36', 'tfidf_37', 'tfidf_38', 'tfidf_39', 'tfidf_40', 'tfidf_41', 'tfidf_42', 'tfidf_43', 'tfidf_44', 'tfidf_45', 'tfidf_46', 'comp_log_price', 'comp_log_pp

## Full-history spline + model-specific Tensor GAM experiment

The smaller combination screen favored **XGBoost + spatiotemporal splines + Tensor GAM**, with **XGBoost + splines + geographic context + Tensor GAM** also promising. This separate comparison evaluates both on the original six chronological validation months using the entire historical training population and the same tuned 700-tree XGBoost specifications.

**Compared arms:** original XGBoost baseline (already fitted); splines only; splines+geo only; splines+Tensor GAM; splines+geo+Tensor GAM.

Each Tensor GAM is calibrated using out-of-time **residuals of its own feature model** on the three prior sold-month cohorts. Prior calibration homes are predicted with full-strength feature models fitted only on sales completed before that month. The script checks provenance before reusing predictions from `outputs/spacetime_full_feature_comparison`; new computations and resumable caches go to `outputs/spacetime_full_spline_gam` and do not touch earlier results.

Run the following cell **after** loading `predictions, models` in the saved-model recovery cell above. Results include comparison to both the original XGBoost and each GAM's respective feature-only baseline. The earlier full-feature comparison is now off by default to avoid repeating it.

**Caveat:** This is another examination of the already-consulted Dec 2025–May 2026 data; it is not an untouched holdout.


In [ ]:
# Run the two full-strength GAM corrections on their corresponding feature bases.
RUN_FULL_SPLINE_GAM_COMPARISON = False  # off-market full test already completed

if RUN_FULL_SPLINE_GAM_COMPARISON:
    from spacetime_full_gam_comparison import compare_full_spline_gam
    from IPython.display import display

    gam_pooled, gam_monthly, gam_by_type, gam_predictions = compare_full_spline_gam(
        baseline_predictions=predictions,
        fitted_models=models,
        months=VALIDATION_MONTHS,
        calibration_months=3,
        calibration_cap=None,
        residual_strength=0.5,
        bootstrap_repeats=500,
        previous_feature_dir="outputs/spacetime_full_feature_comparison",
        output_dir="outputs/spacetime_full_spline_gam",
        resume=True,
    )
    display(gam_pooled)
    display(gam_monthly)
    display(gam_by_type)
    print("Saved results to outputs/spacetime_full_spline_gam/")


## Full-history list-aware XGBoost: spatiotemporal splines, geo and Tensor GAM

This experiment compares **four XGBoost enhancements** on the same six monthly chronological validation cohorts against the **saved list-aware XGBoost**. All XGBoost candidates use the same 700-tree tuned configuration, sale/list-price ratio target and prior-sale history. Their GAMs use only *out-of-time residuals of their own list-aware XGBoost parent model*.

**An independently saved CatBoost + Huber ratio model is optional.** Set `CATBOOST_HUBER_SNAPSHOT` to its saved name if it exists. Otherwise leave it as `None` and run all five XGBoost arms now. The CatBoost reference is never used to fit the new XGBoost or GAM candidates, and is not needed for model selection versus original XGBoost.

- `delta_mdape_pp` / `delta_mape_pp`: paired comparisons against the original XGBoost baseline, always present.
- `delta_parent_mdape_pp` / `delta_parent_mape_pp`: the extra gain of each Tensor GAM over its own XGBoost base.
- `delta_catboost_mdape_pp` / `delta_catboost_mape_pp`: present **only** when the saved CatBoost Huber reference is supplied.

Run after the saved XGBoost recovery cell. An absent Huber file will not stop the experiment. When you recover/retrain it later, set its saved name and **rerun the same cell with `resume=True`**: it will reuse cached XGBoost/GAM results and attach the matched CatBoost comparison. If you have per-home CatBoost validation predictions saved, those can also be compared separately without the CatBoost model.

Outputs: `outputs/spacetime_list_aware_spline_gam_xgb/`. This validation has been consulted already; improvements remain exploratory.


In [6]:
# List-aware XGBoost spline/geo/GAM, optionally benchmarked vs CatBoost Huber.
RUN_LIST_AWARE_SPLINE_GAM = True

if RUN_LIST_AWARE_SPLINE_GAM:
    from spacetime_list_aware_gam import compare_list_aware_spline_gam
    from save_model import list_saved_models
    from IPython.display import display

    print("Saved model snapshots:", list_saved_models())
    # Set to the exact saved Huber model name IF you recover it.
    # None is fully supported: run XGBoost experiments without CatBoost.
    CATBOOST_HUBER_SNAPSHOT = None

    list_pooled, list_monthly, list_by_type, list_predictions = (
        compare_list_aware_spline_gam(
            baseline_predictions=predictions,   # saved XGBoost validation
            fitted_models=models,                # original XGBoost snapshots
            months=VALIDATION_MONTHS,
            catboost_model_name=CATBOOST_HUBER_SNAPSHOT,
            calibration_months=3,
            calibration_cap=None,
            residual_strength=0.5,
            bootstrap_repeats=500,
            output_dir="outputs/spacetime_list_aware_spline_gam_xgb",
            resume=True,
        )
    )
    display(list_pooled)
    display(list_monthly)
    display(list_by_type)
    print("Saved to outputs/spacetime_list_aware_spline_gam_xgb/")


Saved model snapshots: ['CatBoost_Optuna_v1', 'LightGBM', 'LightGBM_Optuna_v1', 'LightGBM_text_binary', 'XGBoost', 'XGBoost_Optuna_v1', 'XGBoost_text_binary', 'catboost_spatiotemporal_900']

2025-12: evaluating 1,238 list-aware homes (XGBoost baseline)
    2025-12 splines: training ratio-XGBoost on 139,931 past sales
    2025-09 splines: training ratio-XGBoost on 133,779 past sales
    2025-10 splines: training ratio-XGBoost on 135,814 past sales
    2025-11 splines: training ratio-XGBoost on 138,064 past sales
    2025-12 splines_tensor_gam: 6,152 prior OOT residuals
    2025-12 splines_geo: training ratio-XGBoost on 139,931 past sales
    2025-09 splines_geo: training ratio-XGBoost on 133,779 past sales
    2025-10 splines_geo: training ratio-XGBoost on 135,814 past sales
    2025-11 splines_geo: training ratio-XGBoost on 138,064 past sales
    2025-12 splines_geo_tensor_gam: 6,152 prior OOT residuals

2026-01: evaluating 1,058 list-aware homes (XGBoost baseline)
    2026-01 splines:

,homes,mape,mdape,median_signed,arm,variant,delta_mdape_pp,delta_mape_pp,mean_paired_ape_delta,homes_improved_fraction,...,delta_mape_ci_high,months_better_mdape,parent_model,delta_parent_mdape_pp,delta_parent_mape_pp,parent_homes_improved_fraction,delta_parent_mdape_ci_low,delta_parent_mdape_ci_high,delta_parent_mape_ci_low,delta_parent_mape_ci_high
0,10069,5.136690,4.077495,0.506349,feature,splines_geo,-0.095356,-0.066285,-0.066285,0.513159,...,-0.026854,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10069,5.129811,4.103349,0.393632,residual,splines_geo_tensor_gam,-0.069501,-0.073163,-0.073163,0.510875,...,-0.033134,4,splines_geo,0.025855,-0.006878,0.497368,-0.006610,0.057595,-0.013404,-0.000542
2,10069,5.123829,4.104579,0.435804,residual,splines_tensor_gam,-0.068271,-0.079145,-0.079145,0.518721,...,-0.043741,4,splines,-0.009878,-0.013035,0.493197,-0.042838,0.030511,-0.019082,-0.006075
3,10069,5.136865,4.114458,0.551463,feature,splines,-0.058393,-0.066110,-0.066110,0.519118,...,-0.031386,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10069,5.202975,4.172850,0.514090,feature,baseline,0.000000,0.000000,0.000000,0.000000,...,0.000000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,homes,mape,mdape,median_signed,month,arm,variant,delta_mdape_pp,delta_mape_pp,delta_parent_mdape_pp,delta_parent_mape_pp
0,1238,4.873582,3.815684,0.939539,2025-12,residual,splines_tensor_gam,-0.256554,-0.211482,-0.097803,-0.100172
1,1238,4.973754,3.913487,1.386406,2025-12,feature,splines,-0.158751,-0.111310,NaN,NaN
2,1238,4.869070,3.957421,0.790047,2025-12,residual,splines_geo_tensor_gam,-0.114817,-0.215993,-0.050813,-0.089494
3,1238,4.958565,4.008234,1.195050,2025-12,feature,splines_geo,-0.064004,-0.126499,NaN,NaN
4,1238,5.085064,4.072238,1.419766,2025-12,feature,baseline,0.000000,0.000000,NaN,NaN
5,1058,4.712554,3.705567,0.394063,2026-01,residual,splines_tensor_gam,-0.418629,-0.181723,-0.089124,-0.067590
6,1058,4.780144,3.794691,0.897976,2026-01,feature,splines,-0.329505,-0.114133,NaN,NaN
7,1058,4.770933,3.810516,0.836120,2026-01,feature,splines_geo,-0.313680,-0.123344,NaN,NaN
8,1058,4.705364,3.903605,0.306867,2026-01,residual,splines_geo_tensor_gam,-0.220592,-0.188913,0.093088,-0.065569
9,1058,4.894277,4.124196,0.823400,2026-01,feature,baseline,0.000000,0.000000,NaN,NaN


,homes,mape,mdape,median_signed,house_type_name,arm,variant,delta_mdape_pp,delta_mape_pp
0,48,5.183326,4.252715,0.939133,Co-Op Apt,residual,splines_tensor_gam,-0.255646,-0.129695
1,48,5.410721,4.376427,1.440993,Co-Op Apt,residual,splines_geo_tensor_gam,-0.131933,0.097700
2,48,5.119943,4.401934,1.154809,Co-Op Apt,feature,splines,-0.106427,-0.193078
3,48,5.313020,4.508360,0.722807,Co-Op Apt,feature,baseline,0.000000,0.000000
4,48,5.353527,4.604586,1.513908,Co-Op Apt,feature,splines_geo,0.096225,0.040507
5,4843,3.906963,3.314288,0.364518,Condo Apt,residual,splines_geo_tensor_gam,-0.012716,-0.080592
6,4843,3.911444,3.318837,0.493780,Condo Apt,feature,splines_geo,-0.008167,-0.076111
7,4843,3.987555,3.327003,0.481660,Condo Apt,feature,baseline,0.000000,0.000000
8,4843,3.925490,3.331786,0.546423,Condo Apt,feature,splines,0.004782,-0.062065
9,4843,3.912421,3.344238,0.394981,Condo Apt,residual,splines_tensor_gam,0.017234,-0.075134


Saved to outputs/spacetime_list_aware_spline_gam_xgb/
